# MoE Load Balancing Loss（MoE 负载均衡损失）

**难度：** Medium

**函数签名：** `moe_load_balance_loss(router_logits, num_experts) -> Tensor`

**参数：**
- `router_logits` — 路由分数 (N_tokens, num_experts)
- `num_experts` — 专家数量

**返回：** 标量辅助损失

**公式：**
- `f_i` = 路由到专家 i 的 token 比例（argmax，不可微）
- `P_i` = 专家 i 的平均路由概率（softmax，可微）
- `L_aux = num_experts * Σ(f_i * P_i)`

**提示：** `f` 是停止梯度的计数项，梯度只通过 `P` 传播

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def moe_load_balance_loss(router_logits, num_experts):
    N = router_logits.shape[0]

    # ---- Step 1: 计算 f_i — token 分配比例 ----
    # argmax 选出每个 token 路由到哪个专家
    # assignments 形状: (N,)，值为 0 到 num_experts-1
    assignments = router_logits.argmax(dim=-1)

    # f[e] = 路由到专家 e 的 token 比例
    # (assignments == e) 生成布尔掩码，.float() 转 0/1，.mean() 求比例
    # 注意: f 不需要梯度！它是不可微的计数项
    f = torch.zeros(num_experts, device=router_logits.device)
    for e in range(num_experts):
        f[e] = (assignments == e).float().mean()

    # ---- Step 2: 计算 P_i — 平均路由概率 ----
    # softmax 将 logits 转为概率分布
    # probs 形状: (N, num_experts)，每行和为 1
    # P = probs.mean(dim=0) 对所有 token 取平均
    # P 形状: (num_experts,)
    # 注意: P 需要梯度！梯度通过 softmax 传播到 router_logits
    probs = torch.softmax(router_logits, dim=-1)
    P = probs.mean(dim=0)

    # ---- Step 3: 计算辅助损失 ----
    # L_aux = num_experts * Σ(f_i * P_i)
    # 为什么这样设计？
    # - 如果分布均匀: f_i = P_i = 1/E → L_aux = E * E * (1/E)^2 = 1.0
    # - 如果分布不均匀（某个专家接收过多 token）: f_i * P_i 变大 → L_aux > 1.0
    # - 梯度只通过 P 传播，推动路由器给低负载专家更高的概率
    return num_experts * (f * P).sum()

In [ ]:
# Uniform routing gives loss = 1.0
# Identical logits -> uniform softmax -> uniform argmax -> f_i = 1/E, P_i = 1/E
# L_aux = E * E * (1/E)^2 = 1.0
num_experts = 4
N_tokens = 100
logits_uniform = torch.zeros(N_tokens, num_experts)
loss_uniform = moe_load_balance_loss(logits_uniform, num_experts)
print(f"Uniform routing loss: {loss_uniform.item():.4f}  (expected 1.0)")

# Collapsed routing gives loss > 1
logits_collapsed = torch.zeros(N_tokens, num_experts)
logits_collapsed[:, 0] = 100.0
loss_collapsed = moe_load_balance_loss(logits_collapsed, num_experts)
print(f"Collapsed routing loss: {loss_collapsed.item():.4f}  (expected > 1.0)")

# Gradient flows through router_logits
torch.manual_seed(0)
logits_grad = torch.randn(16, num_experts, requires_grad=True)
loss_grad = moe_load_balance_loss(logits_grad, num_experts)
loss_grad.backward()
print(f"Gradient exists: {logits_grad.grad is not None}")

In [ ]:
from torch_judge import check
check("moe_load_balance")

## 📝 核心思路总结

1. **f 和 P 的区别**：f 是不可微的计数（argmax），P 是可微的概率（softmax）
2. **均匀分布损失=1**：作为基准，不均匀时损失增大
3. **梯度路径**：loss → P → softmax → router_logits，f 只提供权重